# Supervised GNN Classifier (Direct BCE)

**Hypothesis:** The contrastive objective itself incentivizes tight clustering around extender-code-specific features, causing poor OOD generalization (0–42.6% recall across all SupCon variants). Replacing SupCon loss entirely with direct supervised BCE classification may yield better OOD generalization by not requiring the model to cluster same-class embeddings together.

**Architecture:** Same GAT backbone (3 layers, 4 heads, 256 hidden, residual+LayerNorm, mean pooling) but with a classification head instead of embed+proj heads:
```
graph_embed (256D, unnormalized) → Linear(256, 128) → ReLU → Linear(128, 1) → logit
```

**Key differences from SupCon:**
- BCEWithLogitsLoss instead of SupConLoss
- No L2 normalization on embeddings
- No batch skipping for single-class batches
- Direct model predictions (no linear probe needed for classification)
- 3-way OOD comparison: direct model, linear probe on graph_embed, ECFP4

**Comparison targets:**
- SupCon baseline: 26.4% OOD recall (5K triplets, temp=0.12)
- SupCon+VICReg+ECFP4: 42.6% OOD recall (best ever)
- ECFP4 probe: ~46% OOD recall

## Cell 1: Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Dict, List, Tuple, Iterable

from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import rdchem
RDLogger.DisableLog("rdApp.*")

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, accuracy_score, f1_score, recall_score

# Hyperparameters
BATCH_SIZE = 64
EPOCHS = 10
LR = 3e-4
WEIGHT_DECAY = 1e-4
POS_WEIGHT = 2.0  # Compensates 2:1 non-PKS:PKS ratio from triplet structure
HIDDEN_DIM = 256
GAT_LAYERS = 3
GAT_HEADS = 4
DROPOUT = 0.1
SAMPLE_TRIPLETS = 5000
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Cell 2: Graph Featurization

Reusing the atom/bond featurization scheme from the existing GNN classifier.

In [ ]:
# Atom feature vocabularies
ATOM_TYPES = [1, 5, 6, 7, 8, 9, 14, 15, 16, 17, 35, 53]  # [H, B, C, N, O, F, Si, P, S, Cl, Br, I]
DEGREES = [0, 1, 2, 3, 4, 5]
FORMAL_CHARGES = [-2, -1, 0, 1, 2]
NUM_HS = [0, 1, 2, 3, 4]
HYBRIDIZATIONS = [
    rdchem.HybridizationType.SP,
    rdchem.HybridizationType.SP2,
    rdchem.HybridizationType.SP3,
    rdchem.HybridizationType.SP3D,
    rdchem.HybridizationType.SP3D2,
]
BOND_TYPES = [
    rdchem.BondType.SINGLE,
    rdchem.BondType.DOUBLE,
    rdchem.BondType.TRIPLE,
    rdchem.BondType.AROMATIC,
]

def _build_mapping(values: Iterable) -> Dict:
    return {value: idx for idx, value in enumerate(values)}

ATOM_MAP = _build_mapping(ATOM_TYPES)
DEGREE_MAP = _build_mapping(DEGREES)
CHARGE_MAP = _build_mapping(FORMAL_CHARGES)
NUM_H_MAP = _build_mapping(NUM_HS)
HYB_MAP = {hyb: idx for idx, hyb in enumerate(HYBRIDIZATIONS)}
BOND_MAP = {bond: idx for idx, bond in enumerate(BOND_TYPES)}

EDGE_FEAT_DIM = len(BOND_TYPES) + 1  # +1 for self-loop

def _one_hot(value, mapping: Dict) -> np.ndarray:
    """Create one-hot vector with unknown bucket."""
    size = len(mapping) + 1
    vec = np.zeros(size, dtype=np.float32)
    vec[mapping.get(value, len(mapping))] = 1.0
    return vec

def atom_to_feature(atom: rdchem.Atom) -> np.ndarray:
    """Convert RDKit atom to feature vector (35 dim)."""
    feats = [
        _one_hot(atom.GetAtomicNum(), ATOM_MAP),        # 13 dim
        _one_hot(atom.GetTotalDegree(), DEGREE_MAP),    # 7 dim
        _one_hot(atom.GetFormalCharge(), CHARGE_MAP),   # 6 dim
        _one_hot(atom.GetTotalNumHs(includeNeighbors=True), NUM_H_MAP),  # 6 dim
        _one_hot(atom.GetHybridization(), HYB_MAP),     # 6 dim
        np.array([atom.GetIsAromatic()], dtype=np.float32),  # 1 dim
        np.array([atom.IsInRing()], dtype=np.float32),       # 1 dim
    ]
    return np.concatenate(feats, axis=0)

def bond_to_feature(bond) -> np.ndarray:
    """Convert RDKit bond to feature vector (5 dim)."""
    vec = np.zeros(EDGE_FEAT_DIM, dtype=np.float32)
    if bond is None:
        vec[-1] = 1.0  # self-loop marker
    else:
        vec[BOND_MAP.get(bond.GetBondType(), EDGE_FEAT_DIM - 1)] = 1.0
    return vec

def smiles_to_graph(smiles: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Convert SMILES to graph tensors.
    
    Returns:
        node_feat: [num_atoms, node_feat_dim]
        edge_index: [2, num_edges] (includes self-loops)
        edge_attr: [num_edges, edge_feat_dim]
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    n = mol.GetNumAtoms()
    if n == 0:
        raise ValueError(f"SMILES with no atoms: {smiles}")

    # Node features
    node_feat = np.vstack([atom_to_feature(atom) for atom in mol.GetAtoms()]).astype(np.float32)
    
    # Edge features (bidirectional + self-loops)
    edges: List[Tuple[int, int]] = []
    edge_feat: List[np.ndarray] = []
    for bond in mol.GetBonds():
        u, v = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        feat = bond_to_feature(bond)
        edges.append((u, v))
        edges.append((v, u))
        edge_feat.append(feat)
        edge_feat.append(feat)
    
    # Self-loops
    loop = bond_to_feature(None)
    for i in range(n):
        edges.append((i, i))
        edge_feat.append(loop)
    
    edge_index = np.array(edges, dtype=np.int64).T
    edge_attr = np.vstack(edge_feat).astype(np.float32)
    return node_feat, edge_index, edge_attr

# Test featurization
test_smiles = "CCO"
nf, ei, ea = smiles_to_graph(test_smiles)
print(f"Test SMILES: {test_smiles}")
print(f"Node features shape: {nf.shape}")
print(f"Edge index shape: {ei.shape}")
print(f"Edge attr shape: {ea.shape}")
NODE_FEAT_DIM = nf.shape[1]
print(f"\nNode feature dim: {NODE_FEAT_DIM}")
print(f"Edge feature dim: {EDGE_FEAT_DIM}")

## Cell 3: Dataset and DataLoader

In [ ]:
@dataclass
class GraphSample:
    """Container for a single molecular graph."""
    node_feat: np.ndarray
    edge_index: np.ndarray
    edge_attr: np.ndarray
    label: int


class MolecularGraphDataset(Dataset):
    """Dataset that samples triplets and converts SMILES to graphs on-the-fly."""
    
    def __init__(self, parquet_path: str, num_triplets: int = None, seed: int = SEED):
        df = pd.read_parquet(parquet_path)
        
        if num_triplets is not None:
            # Sample a subset of triplets
            unique_triplets = df['triplet_id'].unique()
            rng = np.random.default_rng(seed)
            sampled_triplets = rng.choice(
                unique_triplets, 
                size=min(num_triplets, len(unique_triplets)), 
                replace=False
            )
            df = df[df['triplet_id'].isin(sampled_triplets)].reset_index(drop=True)
        
        self.smiles = df['smiles'].astype(str).tolist()
        self.labels = df['label'].to_numpy().astype(np.int64)
        
        # Get feature dimensions from first valid molecule
        for smi in self.smiles:
            try:
                nf, _, ea = smiles_to_graph(smi)
                self.node_feat_dim = nf.shape[1]
                self.edge_feat_dim = ea.shape[1]
                break
            except ValueError:
                continue
        else:
            raise RuntimeError("No valid molecules found")
    
    def __len__(self) -> int:
        return len(self.smiles)
    
    def __getitem__(self, idx: int) -> GraphSample:
        nf, ei, ea = smiles_to_graph(self.smiles[idx])
        return GraphSample(nf, ei, ea, int(self.labels[idx]))


def collate_graphs(batch: List[GraphSample]) -> Dict[str, torch.Tensor]:
    """Collate graphs into a batched format."""
    node_feats = []
    edge_indices = []
    edge_attrs = []
    batch_index = []
    labels = []
    offset = 0
    
    for graph in batch:
        x = torch.from_numpy(graph.node_feat)
        ei = torch.from_numpy(graph.edge_index) + offset
        ea = torch.from_numpy(graph.edge_attr)
        n = x.size(0)
        
        node_feats.append(x)
        edge_indices.append(ei)
        edge_attrs.append(ea)
        batch_index.append(torch.full((n,), len(labels), dtype=torch.long))
        labels.append(graph.label)
        offset += n
    
    return {
        'node_feat': torch.cat(node_feats, dim=0),
        'edge_index': torch.cat(edge_indices, dim=1),
        'edge_attr': torch.cat(edge_attrs, dim=0),
        'batch': torch.cat(batch_index, dim=0),
        'labels': torch.tensor(labels, dtype=torch.long),
    }


# Load datasets
print("Loading training data...")
train_ds = MolecularGraphDataset(
    '../data/train/supcon_train.parquet',
    num_triplets=SAMPLE_TRIPLETS,
    seed=SEED
)

print("Loading validation data...")
val_ds = MolecularGraphDataset(
    '../data/val/supcon_val.parquet',
    num_triplets=200,  # Small validation set
    seed=SEED
)

print(f"\nTrain samples: {len(train_ds)}")
print(f"Val samples: {len(val_ds)}")
print(f"Train label distribution: {np.bincount(train_ds.labels)}")
print(f"Val label distribution: {np.bincount(val_ds.labels)}")

# Create data loaders
train_loader = DataLoader(
    train_ds, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_graphs,
    num_workers=0
)

val_loader = DataLoader(
    val_ds, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_graphs,
    num_workers=0
)

## Cell 4: Supervised GNN Classifier

Same GAT backbone as SupCon but with a classification head instead of embed+proj heads:
- `graph_embed` (256D, **unnormalized**) from mean pooling
- Classification head: `Linear(256, 128) → ReLU → Linear(128, 1)`
- Returns `(logit, graph_embed)` for direct prediction + downstream probing

In [ ]:
def edge_softmax(dst: torch.Tensor, scores: torch.Tensor, num_nodes: int) -> torch.Tensor:
    """Compute softmax over edges grouped by destination node.
    
    Args:
        dst: [num_edges] destination node indices
        scores: [num_edges, heads] attention scores
        num_nodes: total number of nodes
    
    Returns:
        [num_edges, heads] attention weights (softmaxed per dst node)
    """
    heads = scores.size(1)
    out = []
    for h in range(heads):
        s = scores[:, h]
        # Compute max per destination for numerical stability
        max_vals = torch.full((num_nodes,), -float('inf'), device=s.device)
        max_vals.scatter_reduce_(0, dst, s, reduce='amax')
        s = s - max_vals[dst]
        exp_s = torch.exp(s)
        # Sum per destination
        denom = torch.zeros(num_nodes, device=s.device).scatter_add_(0, dst, exp_s)
        out.append(exp_s / (denom[dst] + 1e-16))
    return torch.stack(out, dim=1)


class GraphAttentionLayer(nn.Module):
    """Single GAT layer with edge features."""
    
    def __init__(self, in_dim: int, out_dim: int, heads: int, edge_dim: int, dropout: float = 0.0):
        super().__init__()
        self.heads = heads
        self.out_dim = out_dim
        
        self.lin = nn.Linear(in_dim, out_dim * heads, bias=False)
        self.att_src = nn.Parameter(torch.Tensor(heads, out_dim))
        self.att_dst = nn.Parameter(torch.Tensor(heads, out_dim))
        self.bias = nn.Parameter(torch.Tensor(out_dim))
        self.edge_proj = nn.Linear(edge_dim, heads, bias=False)
        self.dropout = dropout
        self.reset_parameters()
    
    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src)
        nn.init.xavier_uniform_(self.att_dst)
        nn.init.zeros_(self.bias)
        nn.init.xavier_uniform_(self.edge_proj.weight)
    
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor) -> torch.Tensor:
        """Forward pass.
        
        Args:
            x: [num_nodes, in_dim] node features
            edge_index: [2, num_edges] edge indices
            edge_attr: [num_edges, edge_dim] edge features
        
        Returns:
            [num_nodes, out_dim] updated node features
        """
        h = self.lin(x)  # [N, heads * out_dim]
        num_nodes = h.size(0)
        h = h.view(num_nodes, self.heads, self.out_dim)  # [N, heads, out_dim]
        
        # Attention scores
        att_src = (h * self.att_src).sum(dim=-1)  # [N, heads]
        att_dst = (h * self.att_dst).sum(dim=-1)  # [N, heads]
        
        src, dst = edge_index
        edge_logits = att_src[src] + att_dst[dst] + self.edge_proj(edge_attr)  # [E, heads]
        edge_logits = F.leaky_relu(edge_logits, 0.2)
        
        # Softmax attention
        edge_alpha = edge_softmax(dst, edge_logits, num_nodes)  # [E, heads]
        edge_alpha = F.dropout(edge_alpha, p=self.dropout, training=self.training)
        
        # Aggregate
        out = h[src] * edge_alpha.unsqueeze(-1)  # [E, heads, out_dim]
        agg = torch.zeros(num_nodes, self.heads, self.out_dim, device=x.device)
        agg.index_add_(0, dst, out)
        
        return agg.mean(dim=1) + self.bias  # [N, out_dim]


class SupervisedGNNClassifier(nn.Module):
    """GNN classifier with direct BCE supervision.
    
    Architecture:
    - Input projection: node_dim -> hidden_dim
    - GAT layers with residual + LayerNorm
    - Mean pooling over nodes -> graph_embed (256D, unnormalized)
    - Classification head: hidden_dim -> 128 -> 1
    """
    
    def __init__(
        self, 
        node_dim: int, 
        edge_dim: int, 
        hidden_dim: int = 256, 
        heads: int = 4, 
        num_layers: int = 3,
        dropout: float = 0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # Input projection
        self.input_proj = nn.Linear(node_dim, hidden_dim)
        
        # GAT layers with LayerNorm
        self.gat_layers = nn.ModuleList()
        self.layer_norms = nn.ModuleList()
        for _ in range(num_layers):
            self.gat_layers.append(
                GraphAttentionLayer(hidden_dim, hidden_dim, heads, edge_dim, dropout)
            )
            self.layer_norms.append(nn.LayerNorm(hidden_dim))
        
        self.dropout = dropout
        
        # Classification head: graph_embed -> logit
        self.cls_head = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
        )
    
    def forward(
        self, 
        node_feat: torch.Tensor, 
        edge_index: torch.Tensor, 
        batch: torch.Tensor, 
        edge_attr: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Forward pass.
        
        Args:
            node_feat: [total_nodes, node_dim] batched node features
            edge_index: [2, total_edges] batched edge indices
            batch: [total_nodes] node-to-graph assignment
            edge_attr: [total_edges, edge_dim] batched edge features
        
        Returns:
            logit: [batch_size, 1] classification logits
            graph_embed: [batch_size, hidden_dim] unnormalized graph embeddings
        """
        # Input projection
        x = self.input_proj(node_feat)
        
        # GAT layers with residual + LayerNorm
        for gat, norm in zip(self.gat_layers, self.layer_norms):
            residual = x
            x = gat(x, edge_index, edge_attr)
            x = norm(x)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + residual  # Residual connection
        
        # Mean pooling over nodes (no L2 normalization)
        num_graphs = int(batch.max().item()) + 1 if batch.numel() > 0 else 0
        graph_embed = torch.zeros(num_graphs, self.hidden_dim, device=x.device)
        counts = torch.zeros(num_graphs, device=x.device)
        graph_embed.index_add_(0, batch, x)
        counts.index_add_(0, batch, torch.ones_like(batch, dtype=x.dtype))
        graph_embed = graph_embed / counts.clamp_min(1.0).unsqueeze(-1)
        
        # Classification head
        logit = self.cls_head(graph_embed)  # [B, 1]
        
        return logit, graph_embed


# Initialize model
model = SupervisedGNNClassifier(
    node_dim=train_ds.node_feat_dim,
    edge_dim=train_ds.edge_feat_dim,
    hidden_dim=HIDDEN_DIM,
    heads=GAT_HEADS,
    num_layers=GAT_LAYERS,
    dropout=DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"\nArchitecture:")
print(f"  Input: {train_ds.node_feat_dim} -> {HIDDEN_DIM}")
print(f"  GAT layers: {GAT_LAYERS} x (hidden={HIDDEN_DIM}, heads={GAT_HEADS})")
print(f"  Classification head: {HIDDEN_DIM} -> 128 -> 1")
print(f"  pos_weight: {POS_WEIGHT}")

## Cell 5: BCE Loss

In [ ]:
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([POS_WEIGHT], device=device)
)

# Test loss computation
logit_test = torch.randn(8, 1, device=device)
label_test = torch.tensor([[0],[0],[0],[1],[1],[1],[0],[1]], dtype=torch.float32, device=device)
loss_test = criterion(logit_test, label_test)
print(f"Test BCE loss: {loss_test.item():.4f}")
print(f"pos_weight={POS_WEIGHT} compensates for 2:1 non-PKS:PKS ratio")

## Cell 6: Training Loop

In [ ]:
import copy

def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch. Returns (loss, accuracy)."""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    num_batches = 0
    
    for batch in loader:
        node_feat = batch['node_feat'].to(device)
        edge_index = batch['edge_index'].to(device)
        edge_attr = batch['edge_attr'].to(device)
        batch_idx = batch['batch'].to(device)
        labels = batch['labels'].float().unsqueeze(1).to(device)  # [B, 1] float for BCE
        
        optimizer.zero_grad()
        
        logits, _ = model(node_feat, edge_index, batch_idx, edge_attr)
        loss = criterion(logits, labels)
        
        if not torch.isfinite(loss):
            continue
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        preds = (logits > 0).float()  # threshold at 0 for logits
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        num_batches += 1
    
    avg_loss = total_loss / max(num_batches, 1)
    accuracy = correct / max(total, 1)
    return avg_loss, accuracy


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    """Evaluate loss and accuracy on a dataset."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    num_batches = 0
    
    for batch in loader:
        node_feat = batch['node_feat'].to(device)
        edge_index = batch['edge_index'].to(device)
        edge_attr = batch['edge_attr'].to(device)
        batch_idx = batch['batch'].to(device)
        labels = batch['labels'].float().unsqueeze(1).to(device)
        
        logits, _ = model(node_feat, edge_index, batch_idx, edge_attr)
        loss = criterion(logits, labels)
        
        if torch.isfinite(loss):
            total_loss += loss.item()
            preds = (logits > 0).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            num_batches += 1
    
    avg_loss = total_loss / max(num_batches, 1)
    accuracy = correct / max(total, 1)
    return avg_loss, accuracy


# Setup optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Training with best-model checkpointing
print("Starting training...")
print(f"{'Epoch':>5} {'Train Loss':>12} {'Train Acc':>12} {'Val Loss':>12} {'Val Acc':>12} {'':>6}")
print("-" * 62)

train_losses = []
val_losses = []
train_accs = []
val_accs = []
best_val_loss = float('inf')
best_model_state = None
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    # Checkpoint best model by val loss
    marker = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        marker = " *best"
    
    print(f"{epoch:>5} {train_loss:>12.4f} {train_acc:>12.4f} {val_loss:>12.4f} {val_acc:>12.4f}{marker}")

# Restore best model
model.load_state_dict(best_model_state)
print(f"\nTraining complete! Restored best model from epoch {best_epoch} (val loss: {best_val_loss:.4f})")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(range(1, EPOCHS + 1), train_losses, label='Train Loss', marker='o', markersize=3)
ax1.plot(range(1, EPOCHS + 1), val_losses, label='Val Loss', marker='s', markersize=3)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('BCE Loss')
ax1.set_title('Supervised GNN: Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2.plot(range(1, EPOCHS + 1), train_accs, label='Train Acc', marker='o', markersize=3)
ax2.plot(range(1, EPOCHS + 1), val_accs, label='Val Acc', marker='s', markersize=3)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Supervised GNN: Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0.5, 1.0])

plt.tight_layout()
plt.show()

print(f"\nFinal train loss: {train_losses[-1]:.4f}, accuracy: {train_accs[-1]:.4f}")
print(f"Final val loss: {val_losses[-1]:.4f}, accuracy: {val_accs[-1]:.4f}")
print(f"Best val loss: {min(val_losses):.4f} (epoch {val_losses.index(min(val_losses)) + 1})")
print(f"Best val accuracy: {max(val_accs):.4f} (epoch {val_accs.index(max(val_accs)) + 1})")

## Cell 7: Direct Model Evaluation

Unlike SupCon, this model can make predictions directly via `sigmoid(logit)`. No linear probe needed for classification (though we still train one on `graph_embed` for comparison).

In [ ]:
@torch.no_grad()
def get_predictions(model, loader, device):
    """Get direct model predictions (sigmoid of logits) and graph embeddings."""
    model.eval()
    all_probs = []
    all_embeds = []
    all_labels = []
    
    for batch in loader:
        node_feat = batch['node_feat'].to(device)
        edge_index = batch['edge_index'].to(device)
        edge_attr = batch['edge_attr'].to(device)
        batch_idx = batch['batch'].to(device)
        
        logits, graph_embed = model(node_feat, edge_index, batch_idx, edge_attr)
        probs = torch.sigmoid(logits).squeeze(1)
        
        all_probs.append(probs.cpu().numpy())
        all_embeds.append(graph_embed.cpu().numpy())
        all_labels.append(batch['labels'].numpy())
    
    return np.concatenate(all_probs), np.vstack(all_embeds), np.concatenate(all_labels)


# Get predictions
print("Getting direct model predictions...")
train_probs, train_embeds, train_labels = get_predictions(model, train_loader, device)
val_probs, val_embeds, val_labels = get_predictions(model, val_loader, device)

train_preds = (train_probs >= 0.5).astype(int)
val_preds = (val_probs >= 0.5).astype(int)

print(f"\nTrain embeddings: {train_embeds.shape}")
print(f"Val embeddings: {val_embeds.shape}")

print("\n--- Direct Model Results ---")
print(f"Train Accuracy: {accuracy_score(train_labels, train_preds):.4f}")
print(f"Train AUPRC:    {average_precision_score(train_labels, train_probs):.4f}")
print(f"Train F1:       {f1_score(train_labels, train_preds):.4f}")
print()
print(f"Val Accuracy:   {accuracy_score(val_labels, val_preds):.4f}")
print(f"Val AUPRC:      {average_precision_score(val_labels, val_probs):.4f}")
print(f"Val F1:         {f1_score(val_labels, val_preds):.4f}")

## Cell 8: Linear Probe on graph_embed (256D)

Train a logistic regression on the unnormalized `graph_embed` (256D) for comparison with the direct model predictions.

In [ ]:
# Train linear classifier on graph_embed (256D, unnormalized)
print("Training linear probe on graph_embed (256D)...")
clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
clf.fit(train_embeds, train_labels)

# Evaluate
train_pred_probe = clf.predict(train_embeds)
train_prob_probe = clf.predict_proba(train_embeds)[:, 1]
val_pred_probe = clf.predict(val_embeds)
val_prob_probe = clf.predict_proba(val_embeds)[:, 1]

print("\n--- Linear Probe Results (graph_embed 256D) ---")
print(f"Train Accuracy: {accuracy_score(train_labels, train_pred_probe):.4f}")
print(f"Train AUPRC:    {average_precision_score(train_labels, train_prob_probe):.4f}")
print(f"Train F1:       {f1_score(train_labels, train_pred_probe):.4f}")
print()
print(f"Val Accuracy:   {accuracy_score(val_labels, val_pred_probe):.4f}")
print(f"Val AUPRC:      {average_precision_score(val_labels, val_prob_probe):.4f}")
print(f"Val F1:         {f1_score(val_labels, val_pred_probe):.4f}")

# Compare direct vs probe
print("\n--- Direct Model vs Linear Probe (Val) ---")
direct_auprc = average_precision_score(val_labels, val_probs)
probe_auprc = average_precision_score(val_labels, val_prob_probe)
print(f"Direct model AUPRC: {direct_auprc:.4f}")
print(f"Linear probe AUPRC: {probe_auprc:.4f}")
print(f"Difference:         {direct_auprc - probe_auprc:+.4f}")

## Cell 9: ECFP4 Fingerprint Baseline

In [ ]:
from rdkit.Chem import AllChem

def smiles_to_ecfp4(smiles: str, nbits: int = 2048) -> np.ndarray:
    """Convert SMILES to ECFP4 fingerprint."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(nbits, dtype=np.float32)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=nbits)
    return np.array(fp, dtype=np.float32)


# Generate fingerprints for training data
print("Generating ECFP4 fingerprints...")
train_fps = np.vstack([smiles_to_ecfp4(smi) for smi in train_ds.smiles])
val_fps = np.vstack([smiles_to_ecfp4(smi) for smi in val_ds.smiles])

print(f"Train fingerprints: {train_fps.shape}")
print(f"Val fingerprints: {val_fps.shape}")

# Train logistic regression on fingerprints
print("\nTraining fingerprint baseline...")
clf_fp = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
clf_fp.fit(train_fps, train_ds.labels)

# Evaluate
train_pred_fp = clf_fp.predict(train_fps)
train_prob_fp = clf_fp.predict_proba(train_fps)[:, 1]
val_pred_fp = clf_fp.predict(val_fps)
val_prob_fp = clf_fp.predict_proba(val_fps)[:, 1]

print("\n--- Linear Probe Results (ECFP4 Fingerprints) ---")
print(f"Train Accuracy: {accuracy_score(train_ds.labels, train_pred_fp):.4f}")
print(f"Train AUPRC:    {average_precision_score(train_ds.labels, train_prob_fp):.4f}")
print(f"Train F1:       {f1_score(train_ds.labels, train_pred_fp):.4f}")
print()
print(f"Val Accuracy:   {accuracy_score(val_ds.labels, val_pred_fp):.4f}")
print(f"Val AUPRC:      {average_precision_score(val_ds.labels, val_prob_fp):.4f}")
print(f"Val F1:         {f1_score(val_ds.labels, val_pred_fp):.4f}")

## Cell 10: OOD Benchmark — 3-Way Comparison

Evaluate on 678 PKS molecules from held-out extender codes (butylmalonyl, hexylmalonyl, isobutylmalonyl, d-isobutylmalonyl, dcp). All OOD molecules are true PKS (label=1), so the key metric is **recall**.

Three methods compared:
1. **Direct model**: `sigmoid(logit)` from the supervised GNN
2. **Linear probe**: LogisticRegression on `graph_embed` (256D)
3. **ECFP4 baseline**: LogisticRegression on ECFP4 fingerprints (2048D)

In [ ]:
from pathlib import Path

OOD_SMILES_PATH = "../data/processed/eval_pks_products_1_ext_no_stereo_butmal_hexmal_isobutmal_d-isobutmal_dcp_SMILES.txt"

ood_path = Path(OOD_SMILES_PATH)
if not ood_path.exists():
    print(f"OOD SMILES file not found at {OOD_SMILES_PATH}")
    print("Run scripts/11_generate_ood_eval_set.py first.")
else:
    ood_smiles = [l.strip() for l in ood_path.read_text().splitlines() if l.strip()]
    print(f"Loaded {len(ood_smiles)} OOD PKS molecules (all label=1)")

    # --- Extract GNN predictions and embeddings for OOD ---
    print("\nExtracting GNN predictions and embeddings for OOD molecules...")
    ood_direct_probs = []
    ood_gnn_embeds = []
    ood_gnn_failed = []
    model.eval()
    with torch.no_grad():
        for smi in ood_smiles:
            try:
                nf, ei, ea = smiles_to_graph(smi)
            except ValueError:
                ood_gnn_failed.append(smi)
                continue
            nf_t = torch.from_numpy(nf).to(device)
            ei_t = torch.from_numpy(ei).to(device)
            ea_t = torch.from_numpy(ea).to(device)
            bt_t = torch.zeros(nf.shape[0], dtype=torch.long, device=device)
            logit, graph_embed = model(nf_t, ei_t, bt_t, ea_t)
            ood_direct_probs.append(torch.sigmoid(logit).cpu().numpy().item())
            ood_gnn_embeds.append(graph_embed.cpu().numpy().squeeze(0))
    
    ood_direct_probs = np.array(ood_direct_probs)
    ood_gnn_embeds = np.vstack(ood_gnn_embeds)
    if ood_gnn_failed:
        print(f"  WARNING: {len(ood_gnn_failed)} SMILES failed graph conversion")
    print(f"  GNN embeddings: {ood_gnn_embeds.shape}")

    # --- Compute ECFP4 fingerprints for OOD ---
    print("Computing ECFP4 fingerprints for OOD molecules...")
    ood_fps = np.vstack([smiles_to_ecfp4(smi) for smi in ood_smiles])
    print(f"  ECFP4 fingerprints: {ood_fps.shape}")

    # --- Run all three methods ---
    ood_probe_probs = clf.predict_proba(ood_gnn_embeds)[:, 1]
    ood_fp_probs = clf_fp.predict_proba(ood_fps)[:, 1]

    # --- Results ---
    def ood_stats(probs):
        return float(np.mean(probs >= 0.5)), float(np.mean(probs)), float(np.median(probs))

    direct_recall, direct_mean, direct_median = ood_stats(ood_direct_probs)
    probe_recall, probe_mean, probe_median = ood_stats(ood_probe_probs)
    fp_recall, fp_mean, fp_median = ood_stats(ood_fp_probs)

    print("\n" + "=" * 76)
    print(f"OOD Benchmark: {len(ood_smiles)} held-out PKS molecules (all label=1)")
    print("=" * 76)
    print(f"  {'Method':<25} {'Recall':>10} {'Mean prob':>12} {'Median prob':>14}")
    print("  " + "-" * 63)
    print(f"  {'Direct model (sigmoid)':<25} {direct_recall:>10.4f} {direct_mean:>12.4f} {direct_median:>14.4f}")
    print(f"  {'Linear probe (256D)':<25} {probe_recall:>10.4f} {probe_mean:>12.4f} {probe_median:>14.4f}")
    print(f"  {'ECFP4 fingerprint':<25} {fp_recall:>10.4f} {fp_mean:>12.4f} {fp_median:>14.4f}")
    print("=" * 76)
    
    print("\n--- Comparison with prior experiments (all 5K triplets, 10 epochs) ---")
    print(f"  SupCon baseline:          26.4% OOD recall")
    print(f"  SupCon+VICReg+ECFP4:      42.6% OOD recall (best SupCon variant)")
    print(f"  ECFP4 probe:              ~46% OOD recall")
    print(f"  Supervised GNN (direct):  {direct_recall*100:.1f}% OOD recall")
    print(f"  Supervised GNN (probe):   {probe_recall*100:.1f}% OOD recall")

## Cell 11: OOD Probability Distribution Histograms

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Direct model
ax = axes[0]
ax.hist(ood_direct_probs, bins=50, alpha=0.8, color='tab:green', edgecolor='black', linewidth=0.5)
ax.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Threshold')
recall_pct = np.mean(ood_direct_probs >= 0.5) * 100
ax.set_title(f'Direct Model (sigmoid)\nOOD Recall: {recall_pct:.1f}%')
ax.set_xlabel('P(PKS)')
ax.set_ylabel('Count')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Linear probe on graph_embed
ax = axes[1]
ax.hist(ood_probe_probs, bins=50, alpha=0.8, color='tab:blue', edgecolor='black', linewidth=0.5)
ax.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Threshold')
recall_pct = np.mean(ood_probe_probs >= 0.5) * 100
ax.set_title(f'Linear Probe (graph_embed 256D)\nOOD Recall: {recall_pct:.1f}%')
ax.set_xlabel('P(PKS)')
ax.set_ylabel('Count')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: ECFP4
ax = axes[2]
ax.hist(ood_fp_probs, bins=50, alpha=0.8, color='tab:orange', edgecolor='black', linewidth=0.5)
ax.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Threshold')
recall_pct = np.mean(ood_fp_probs >= 0.5) * 100
ax.set_title(f'ECFP4 Fingerprint\nOOD Recall: {recall_pct:.1f}%')
ax.set_xlabel('P(PKS)')
ax.set_ylabel('Count')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f'OOD PKS Probability Distributions ({len(ood_smiles)} held-out molecules)', 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\nDistribution shape interpretation:")
print("  - Bimodal (mass at 0 and 1): model is confident but wrong on many OOD")
print("  - Collapsed near 0: model confidently predicts non-PKS for OOD (SupCon failure mode)")
print("  - Spread across [0,1]: model is uncertain on OOD (potentially recoverable with calibration)")
print("  - Peaked near 1: model correctly identifies OOD as PKS")

## Cell 12: Summary

### Results

| Method | Val AUPRC | OOD Recall | Notes |
|--------|-----------|------------|-------|
| SupCon baseline | ~0.998 | 26.4% | 5K triplets, temp=0.12 |
| SupCon+VICReg+ECFP4 | 0.880 | 42.6% | Best SupCon variant |
| ECFP4 probe | ~0.86 | ~46% | Fixed hashes, no learned collapse |
| **Supervised GNN (direct)** | **TBD** | **TBD** | This notebook |
| **Supervised GNN (probe)** | **TBD** | **TBD** | This notebook |

### Interpretation

**If OOD recall improves over SupCon:** The contrastive objective was indeed the bottleneck. Direct supervision doesn't require same-class embeddings to cluster tightly, so the backbone can encode more diverse structural features.

**If OOD recall is similar or worse:** The OOD gap is a fundamental property of learned GNN representations (not specific to the loss function). The GAT backbone itself may be learning extender-code-specific features because they're the most discriminative, regardless of the training objective.

### Key Differences from SupCon
- No L2 normalization on embeddings (BCE doesn't need unit-sphere geometry)
- No batch skipping for single-class batches (BCE works with any class mix)
- Classification head replaces embed+proj heads (fewer parameters)
- Direct predictions possible (no linear probe needed)